# Imports

In [1]:
!pip install retina-face

In [2]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import os
from pathlib import Path

from typing import List, Union

import torch #PyTorch

from tqdm.notebook import tqdm

import cv2 #OpenCV
from retinaface import RetinaFace

# Setup

In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Config
# Path to dataset (Each group member may need to modify this)
DATASET_RAW_FILEPATH_IN_DRIVE = "/content/drive/MyDrive/COMP-SCI 5530 - Principles of Data Science/Project/data_raw/deepfake-and-real-images.zip"

In [5]:
# Copy data clean zip from Google Drive
!cp "{DATASET_RAW_FILEPATH_IN_DRIVE}" "/content/data_raw.zip"

In [6]:
# Unzip data raw zip
!unzip "/content/data_raw.zip"

Streaming output truncated to the last 5000 lines.
  inflating: Dataset/Validation/Real/real_5499.jpg  
  inflating: Dataset/Validation/Real/real_55.jpg  
  inflating: Dataset/Validation/Real/real_550.jpg  
  inflating: Dataset/Validation/Real/real_5500.jpg  
  inflating: Dataset/Validation/Real/real_5501.jpg  
  inflating: Dataset/Validation/Real/real_5502.jpg  
  inflating: Dataset/Validation/Real/real_5503.jpg  
  inflating: Dataset/Validation/Real/real_5504.jpg  
  inflating: Dataset/Validation/Real/real_5505.jpg  
  inflating: Dataset/Validation/Real/real_5506.jpg  
  inflating: Dataset/Validation/Real/real_5507.jpg  
  inflating: Dataset/Validation/Real/real_5508.jpg  
  inflating: Dataset/Validation/Real/real_5509.jpg  
  inflating: Dataset/Validation/Real/real_551.jpg  
  inflating: Dataset/Validation/Real/real_5510.jpg  
  inflating: Dataset/Validation/Real/real_5511.jpg  
  inflating: Dataset/Validation/Real/real_5512.jpg  
  inflating: Dataset/Validation/Real/real_5513.jpg  

In [7]:
# Print GPU info
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU: NVIDIA L4


In [8]:
# Attempt to run on GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Running on:", device)

Running on: cuda


In [17]:
RetinaFace.build_model()

# Helper Fuctions

In [18]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

def get_faces(img_path: str) -> List[np.ndarray]:
    faces = RetinaFace.extract_faces(img_path = img_path, align = True)

    return faces

# Cleaning Data

In [20]:
raw_dir_root = "/content/Dataset"
clean_dir_root = "/content/data_clean"

counter = 0

# Get total num files in raw data
total_files = 0
for root, dirs, files in os.walk(raw_dir_root):
  total_files += len(files)

with tqdm(total = total_files, desc="Cleaning Data") as pbar:
  for folder in os.listdir(raw_dir_root):
    folder_path = os.path.join(raw_dir_root, folder)
    for subfolder in os.listdir(folder_path):
      subfolder_path = os.path.join(folder_path, subfolder)
      save_path = f"{clean_dir_root}/{folder}/{subfolder}"
      os.makedirs(save_path, exist_ok=True)
      for img_name in os.listdir(subfolder_path):
        img_path = os.path.join(subfolder_path, img_name)

        faces = get_faces(img_path = img_path)
        if not faces:
          pbar.update(1)
          continue
        for i, face in enumerate(faces):
          if face is None or face.size == 0:
            continue
          clean_img_path = os.path.join(save_path, f"{img_name}_{i}")
          cv2.imwrite(clean_img_path, face)

        pbar.update(1)

Cleaning Data:   0%|          | 0/190335 [00:00<?, ?it/s]

In [21]:
!zip -r "data_clean.zip" "data_clean"

Streaming output truncated to the last 5000 lines.
  adding: data_clean/Train/Fake/fake_51216.jpg_0 (deflated 2%)
  adding: data_clean/Train/Fake/fake_20451.jpg_0 (deflated 3%)
  adding: data_clean/Train/Fake/fake_23564.jpg_0 (deflated 2%)
  adding: data_clean/Train/Fake/fake_52180.jpg_0 (deflated 2%)
  adding: data_clean/Train/Fake/fake_14623.jpg_1 (deflated 2%)
  adding: data_clean/Train/Fake/fake_6036.jpg_0 (deflated 3%)
  adding: data_clean/Train/Fake/fake_30158.jpg_0 (deflated 3%)
  adding: data_clean/Train/Fake/fake_29598.jpg_0 (deflated 3%)
  adding: data_clean/Train/Fake/fake_15116.jpg_0 (deflated 3%)
  adding: data_clean/Train/Fake/fake_46947.jpg_0 (deflated 2%)
  adding: data_clean/Train/Fake/fake_34492.jpg_0 (deflated 3%)
  adding: data_clean/Train/Fake/fake_2002.jpg_0 (deflated 2%)
  adding: data_clean/Train/Fake/fake_10953.jpg_0 (deflated 2%)
  adding: data_clean/Train/Fake/fake_10916.jpg_0 (deflated 2%)
  adding: data_clean/Train/Fake/fake_16158.jpg_1 (deflated 4%)
  addi

In [22]:
!cp "/content/data_clean.zip" "/content/drive/MyDrive"

In [23]:
from google.colab import runtime
runtime.unassign()